# DS200 - Lab 2


Khởi tạo Spark Session


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType, LongType, DateType

spark = SparkSession.builder \
    .appName("Lab 3") \
    .master("local[*]") \
    .getOrCreate()

Load Data


In [2]:
order_df = spark.read.format("csv").options(header="True", delimiter=";", inferSchema="True").load("Orders.csv")
product_df = spark.read.format("csv").options(header="True", delimiter=";", inferSchema="True").load("Products.csv")
order_item_df = spark.read.format("csv").options(header="True", delimiter=";", inferSchema="True").load("Order_Items.csv")
customer_df = spark.read.format("csv").options(header="True", delimiter=";", inferSchema="True").load("Customer_List.csv")
order_review_df = spark.read.format("csv").options(header="True", delimiter=";", inferSchema="True").load("Order_Reviews.csv")

1. Hãy đọc dữ liệu từ các file csv, sử dụng tự suy ra kiểu dữ liệu cho mỗi cột.


In [3]:
order_df.printSchema()
product_df.printSchema()
order_item_df.printSchema()
customer_df.printSchema()
order_review_df.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)

root
 |-- Product_ID: string (nullable = true)
 |-- Product_Category_Name: string (nullable = true)
 |-- Product_Weight_Gr: integer (nullable = true)
 |-- Product_Length_Cm: integer (nullable = true)
 |-- Product_Height_Cm: integer (nullable = true)
 |-- Product_Width_Cm: integer (nullable = true)

root
 |-- Order_ID: string (nullable = true)
 |-- Order_Item_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Seller_ID: string (nullable = true)
 |-- Shipping_Limit_Date: timestamp (nullable = true)
 |-- Price: double (nullable = tr

In [4]:
order_df.show(5)
product_df.show(5)
order_item_df.show(5)
customer_df.show(5)
order_review_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            Order_ID|     Customer_Trx_ID|Order_Status|Order_Purchase_Timestamp|  Order_Approved_At|Order_Delivered_Carrier_Date|Order_Delivered_Customer_Date|Order_Estimated_Delivery_Date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2023-10-02 10:56:00|2023-10-02 11:07:00|         2023-10-04 19:55:00|          2023-10-10 21:25:00|          2023-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2024-07-24 20:41:00|2024-07-26 03:24:00|         2024-07-26 14:31:00|          2024-08-07 15:27:00|          2024-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

2. Thống kê tổng số đơn hàng, số lượng khách hàng và người bán.


In [5]:
# Thống kê tổng số đơn hàng, số lượng khách hàng và người bán
total_orders = order_df.count()
total_customers = customer_df.select("Customer_Trx_ID").distinct().count()
total_sellers = order_item_df.select("seller_id").distinct().count()

print(f"Tổng số đơn hàng: {total_orders}")
print(f"Số lượng khách hàng: {total_customers}")
print(f"Số lượng người bán: {total_sellers}")

Tổng số đơn hàng: 99441
Số lượng khách hàng: 99442
Số lượng người bán: 3095


3. Phân tích số lượng đơn hàng theo quốc gia, sắp xếp theo thứ tự giảm dần.


In [6]:
from pyspark.sql.functions import count
from pyspark.sql.functions import countDistinct

orders_by_country = order_df.join(customer_df, order_df.Customer_Trx_ID == customer_df.Customer_Trx_ID, "inner") \
    .groupBy(customer_df.Customer_Country) \
    .agg(count("Order_ID").alias("Total_Orders")) \
    .orderBy("Total_Orders", ascending=False)

orders_by_country.show()

+----------------+------------+
|Customer_Country|Total_Orders|
+----------------+------------+
|         Germany|       41754|
|          France|       12848|
|     Netherlands|       11629|
|         Belgium|        5464|
|         Austria|        5043|
|     Switzerland|        3640|
|  United Kingdom|        3382|
|          Poland|        2139|
|         Czechia|        2034|
|           Italy|        2025|
|           Spain|        1651|
|        Portugal|        1336|
|          Sweden|         975|
|         Denmark|         905|
|          Serbia|         746|
|          Norway|         716|
|        Slovakia|         534|
|        Slovenia|         495|
|          Turkey|         485|
|          Greece|         412|
+----------------+------------+
only showing top 20 rows


4. Phân tích số lượng đơn hàng nhóm theo năm, tháng đặt hàng (Hiển thị theo năm
   tăng dần, tháng giảm dần)


In [7]:
from pyspark.sql.functions import year, month, desc

order_df.withColumn("order_year", year("Order_Purchase_Timestamp")) \
        .withColumn("order_month", month("Order_Purchase_Timestamp")) \
        .groupBy("order_year", "order_month") \
        .count() \
        .orderBy("order_year", desc("order_month")) \
        .show()


+----------+-----------+-----+
|order_year|order_month|count|
+----------+-----------+-----+
|      2022|         12|    1|
|      2022|         10|  324|
|      2022|          9|    4|
|      2023|         12| 5673|
|      2023|         11| 7544|
|      2023|         10| 4631|
|      2023|          9| 4285|
|      2023|          8| 4331|
|      2023|          7| 4026|
|      2023|          6| 3245|
|      2023|          5| 3700|
|      2023|          4| 2404|
|      2023|          3| 2682|
|      2023|          2| 1780|
|      2023|          1|  800|
|      2024|         10|    4|
|      2024|          9|   16|
|      2024|          8| 6512|
|      2024|          7| 6292|
|      2024|          6| 6167|
+----------+-----------+-----+
only showing top 20 rows


5. Thống kê điểm đánh giá trung bình, số lượng đánh giá theo từng mức (ví dụ: 1 đến
   5).


In [8]:
from pyspark.sql.functions import col

order_review_df_category = order_review_df \
    .filter(col("Review_Score").isin("1", "2", "3", "4", "5")) \
    .groupBy("Review_Score") \
    .count() \
    .orderBy("Review_Score", ascending=True)
    
order_review_df_category.show()

+------------+-----+
|Review_Score|count|
+------------+-----+
|           1|11424|
|           2| 3151|
|           3| 8179|
|           4|19141|
|           5|57328|
+------------+-----+



6. Tính doanh thu (giá sản phẩm + phí vận chuyển) trong năm 2024 và nhóm theo
   danh mục sản phẩm


In [9]:
from pyspark.sql.functions import avg

avg_review_score = order_review_df \
    .filter(col("Review_Score").isin("1", "2", "3", "4", "5")) \
    .withColumn("Review_Score", col("Review_Score").cast("int")) \
    .groupBy() \
    .agg(avg("Review_Score").alias("Average_Review_Score"))
    
avg_review_score.show()

+--------------------+
|Average_Review_Score|
+--------------------+
|  4.0864214950162765|
+--------------------+



In [10]:
from pyspark.sql.functions import col, year, sum

order_item_with_full_price = order_item_df \
    .withColumn("Full_Price", col("Price") + col("Freight_Value"))

order_item_with_full_price = order_item_with_full_price \
    .join(product_df, order_item_with_full_price.Product_ID == product_df.Product_ID, "inner") \
    .join(order_df, order_item_with_full_price.Order_ID == order_df.Order_ID, "inner")

order_item_with_full_price = order_item_with_full_price \
    .withColumn("purchase_year", year("Order_Purchase_Timestamp")) \
    .filter(col("purchase_year") == 2024)

revenue_by_category = order_item_with_full_price \
    .groupBy("Product_Category_Name") \
    .agg(sum("Full_Price").alias("Doanh_Thu")) \
    .orderBy("Doanh_Thu", ascending=False)

revenue_by_category.show()

+---------------------+------------------+
|Product_Category_Name|         Doanh_Thu|
+---------------------+------------------+
|        Health_Beauty| 885191.1200000007|
|        Watches_Gifts| 771986.7499999991|
|       Bed_Bath_Table| 650794.6999999994|
|       Sports_Leisure| 621999.3399999996|
| Computers_Accesso...| 594771.0400000003|
|           Housewares| 491576.9600000005|
|      Furniture_Decor|476466.12999999983|
|                 Auto|404210.56999999995|
|                 Baby|299052.56000000006|
|           Cool_Stuff|273910.05000000005|
|         Garden_Tools|259068.31999999983|
|            Telephony|217452.12999999963|
|            Perfumery|204562.53999999986|
|                 Toys|200634.06999999998|
|     Office_Furniture| 181745.7300000001|
|           Stationery|         164743.85|
|             Pet_Shop|152804.93999999992|
| Construction_Tool...|         141187.34|
|          Electronics|134265.45000000007|
|  Musical_Instruments|121476.31000000001|
+----------

7. Xác định sản phẩm có số lượng bán ra cao nhất và tính điểm đánh giá trung bình
   cho từng sản phẩm


In [11]:
product_order_summary = order_item_df \
    .join(order_review_df, order_item_df.Order_ID == order_review_df.Order_ID, "inner") \
    .groupBy("Product_ID") \
    .agg(
        count("Product_ID").alias("So_Luong"),
        avg("Review_Score").alias("Diem_TB")
    ) \
    .orderBy("So_Luong", ascending=False)
        
product_order_summary.show()

+--------------------+--------+------------------+
|          Product_ID|So_Luong|           Diem_TB|
+--------------------+--------+------------------+
|aca2eb7d00ea1a7b8...|     524| 4.019083969465649|
|422879e10f4668299...|     486|3.9465020576131686|
|99a4788cb24856965...|     482|3.8983402489626555|
|389d119b48cf3043d...|     391| 4.117647058823529|
|368c6c730842d7801...|     388| 3.922680412371134|
|53759a2ecddad2bb8...|     373| 3.868632707774799|
|d1c427060a0f73f6b...|     340| 4.194117647058824|
|53b36df67ebb7c415...|     320|          4.190625|
|154e7e31ebfa09220...|     292| 4.315068493150685|
|3dd2a17168ec895c7...|     272| 4.209558823529412|
|2b4609f8948be1887...|     269| 4.070631970260223|
|7c1bd920dbdf22470...|     235| 3.876595744680851|
|a62e25e09e05e6faf...|     225|3.8577777777777778|
|bb50f2e236e5eea01...|     196| 4.224489795918367|
|5a848e4ab52fd5445...|     195|4.1179487179487175|
|e0d64dcfaa3b6db5c...|     193|  3.77720207253886|
|e53e557d5a159f5aa...|     185|

8. Tính toán hiệu số giữa ngày giao hàng thực tế (Order_Delivered_Carrier_Date) và
   ngày giao hàng dự kiến (ví dụ: Shipping_Limit_Date từ bảng Order_Items) để
   đánh giá hiệu suất giao hàng


In [12]:
from pyspark.sql.functions import datediff, col, avg, when

# Join order_df với order_item_df để lấy thông tin ngày giao hàng và ngày dự kiến
delivery_performance = order_df \
    .join(order_item_df, order_df.Order_ID == order_item_df.Order_ID, "inner") \
    .select(
        order_df.Order_ID,
        order_df.Order_Delivered_Carrier_Date,
        order_item_df.Shipping_Limit_Date
    ) \
    .withColumn("Delivery_Delay_Days", 
                datediff(col("Order_Delivered_Carrier_Date"), col("Shipping_Limit_Date")))

# Thống kê hiệu suất giao hàng
delivery_stats = delivery_performance \
    .groupBy() \
    .agg(
        avg("Delivery_Delay_Days").alias("Avg_Delay_Days"),
        count(when(col("Delivery_Delay_Days") > 0, 1)).alias("Late_Deliveries"),
        count(when(col("Delivery_Delay_Days") <= 0, 1)).alias("On_Time_Deliveries")
    )

print("Chi tiết hiệu suất giao hàng:")
delivery_performance.show(10)
print("\nThống kê tổng quan:")
delivery_stats.show()

Chi tiết hiệu suất giao hàng:
+--------------------+----------------------------+-------------------+-------------------+
|            Order_ID|Order_Delivered_Carrier_Date|Shipping_Limit_Date|Delivery_Delay_Days|
+--------------------+----------------------------+-------------------+-------------------+
|00010242fe8c5a6d1...|         2023-09-19 18:34:00|2023-09-19 09:45:00|                  0|
|00018f77f2f0320c5...|         2023-05-04 14:35:00|2023-05-03 11:05:00|                  1|
|000229ec398224ef6...|         2024-01-16 12:36:00|2024-01-18 14:48:00|                 -2|
|00024acbcdf0a6daa...|         2024-08-10 13:28:00|2024-08-15 10:10:00|                 -5|
|00042b26cf59d7ce6...|         2023-02-16 09:46:00|2023-02-13 13:57:00|                  3|
|00048cc3ae777c65d...|         2023-05-17 11:05:00|2023-05-23 03:55:00|                 -6|
|00054e8431b9d7675...|         2023-12-12 01:07:00|2023-12-14 12:10:00|                 -2|
|000576fe39319847c...|         2024-07-05 12:15:00

9. Nhóm khách hàng dựa trên số lượng đơn hàng, giá trị trung bình của đơn hàng và
   tần suất mua sắm.


10. Xếp hạng các seller dựa trên tổng doanh thu và số lượng đơn hàng bán được.


In [14]:
from pyspark.sql.functions import sum, count, rank
from pyspark.sql.window import Window

# Tính doanh thu và số đơn hàng cho mỗi seller
seller_performance = order_item_df \
    .withColumn("Total_Price", col("Price") + col("Freight_Value")) \
    .groupBy("seller_id") \
    .agg(
        sum("Total_Price").alias("Total_Revenue"),
        count("Order_ID").alias("Total_Orders")
    )

# Tạo window để xếp hạng
window_revenue = Window.orderBy(col("Total_Revenue").desc())
window_orders = Window.orderBy(col("Total_Orders").desc())

# Xếp hạng sellers
seller_ranking = seller_performance \
    .withColumn("Revenue_Rank", rank().over(window_revenue)) \
    .withColumn("Orders_Rank", rank().over(window_orders)) \
    .withColumn("Overall_Score", (col("Revenue_Rank") + col("Orders_Rank")) / 2) \
    .orderBy("Overall_Score")

print("Top 20 Sellers:")
seller_ranking.show(20)

# Hiển thị top 10 theo doanh thu
print("\nTop 10 Sellers theo doanh thu:")
seller_ranking.orderBy("Total_Revenue", ascending=False).show(10)

Top 20 Sellers:
+--------------------+------------------+------------+------------+-----------+-------------+
|           seller_id|     Total_Revenue|Total_Orders|Revenue_Rank|Orders_Rank|Overall_Score|
+--------------------+------------------+------------+------------+-----------+-------------+
|4a3ca9315b744ce9f...|235539.95999999967|        1987|           4|          2|          3.0|
|7c67e1448b00f6e96...|239536.43999999994|        1364|           2|          8|          5.0|
|da8622b14eb17ae28...|185192.32000000015|        1551|           6|          5|          5.5|
|6560211a19b47992c...|151265.76999999967|        2033|          11|          1|          6.0|
|4869f7a5dfa277a7d...| 249640.6999999999|        1156|           1|         11|          6.0|
|1f50f920176fa81da...|142104.97999999986|        1931|          12|          3|          7.5|
|1025f0e2d44d7041d...|         172860.69|        1428|           8|          7|          7.5|
|955fee9216a65b617...| 160602.6800000002|   

In [15]:
# Stop Spark
spark.stop()